In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageOps
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "common/config.yaml").exists())
sys.path.insert(0, str(ROOT))

from phase1.pair_b.src.composite import load_cutouts, paste
from common import load_config, resolve
from common.data import get_transform, normalize, set_seed
from common.model import load_model, get_device, split_model

In [ ]:
P1 = ROOT / "phase1/pair_b"

GROUPS = ["elephant", "deer", "pig/peccary"]
PER_SPLIT_HABITAT = 4 
K = 5  # num of neighbors
SEED = 42

In [ ]:
def prepare_composites():
    cfg = load_config()
    set_seed(SEED)
    device = get_device(cfg)
    backgrounds = pd.read_parquet(P1 / "results/backgrounds.parquet")
    pool = backgrounds.loc[~backgrounds.infrared].sample(frac=1, random_state=SEED)
    backgrounds = (pool.drop_duplicates("location").groupby(["split", "habitat"])
                   .head(PER_SPLIT_HABITAT).sort_values("bg_id"))
    
    all_cutouts = pd.read_csv(P1 / "cutouts/cutouts.csv")
    chosen = all_cutouts.loc[all_cutouts.group.isin(GROUPS)]
    original = pd.read_parquet(P1 / "results/composites.parquet")
    original = original.loc[(original.row == 1) & (original.col == 1)]  # centre placement
    original = original.set_index(["group", "variant", "bg_id"])

    names = pd.read_csv(ROOT / "data/species_groups.csv").set_index("y").common_name.to_dict()
    image_dir = resolve(cfg["paths"]["data_dir"]) / "train"
    transform = get_transform(cfg)
    background_images = []
    for filename in backgrounds.filename:
        with Image.open(image_dir / filename) as image:
            background_images.append(transform(image.convert("RGB")))
    background_images = torch.stack(background_images)

    cutouts = load_cutouts()
    images, records = [], []
    for animal in chosen.itertuples():
        batch = paste(background_images, cutouts[animal.group, animal.variant], 1, 1)
        for j, bg in enumerate(backgrounds.itertuples()):
            prediction = original.loc[(animal.group, animal.variant, bg.bg_id)]
            images.append(batch[j])
            records.append(dict(variant=animal.variant, bg_id=bg.bg_id, y=prediction.y,
                                pred_y=prediction.pred_y, habitat=bg.habitat, split=bg.split))
    images = torch.stack(images)
    records = pd.DataFrame(records)

    backbone, _ = split_model(load_model(cfg))
    parts = []
    with torch.no_grad():
        for start in range(0, len(images), 8):
            features = backbone(normalize(images[start:start+8].to(device)))
            parts.append(features.cpu().numpy())

    metadata = pd.read_parquet(resolve(cfg["paths"]["predictions"]),
        columns=["id", "split", "label", "prediction", "seq_id", "filename"])
    embeddings = np.load(resolve(cfg["paths"]["embeddings"]), mmap_mode="r")
    
    assert len(metadata) == len(embeddings), "Prediction/embedding row alignment mismatch."
    return records, images, np.concatenate(parts), metadata, embeddings, all_cutouts, names, image_dir

In [ ]:
def find_neighbors(queries, metadata, embeddings, all_cutouts):
    sequence = metadata.seq_id.fillna(metadata.id)
    source_sequences = sequence[metadata.id.isin(all_cutouts.image_id)]

    # keep only training split and images not in the cut-out sequences
    eligible = metadata.split.eq("train") & ~sequence.isin(source_sequences)

    training_rows = np.flatnonzero(eligible)
    codes, sequences = pd.factorize(sequence.iloc[training_rows], sort=True)

    # normalize
    queries = queries / np.maximum(np.linalg.norm(queries, axis=1, keepdims=True), 1e-12)

    scores = np.full((len(queries), len(sequences)), -np.inf, np.float32)
    image_rows = np.full(scores.shape, -1)

    # keep the closest image within each image sequence, then rank sequences
    for start in range(0, len(training_rows), 4096):
        rows = training_rows[start:start+4096]
        codes_block = codes[start:start+4096]
        features = embeddings[rows]
        norms = np.linalg.norm(features, axis=1, keepdims=True)

        # cosine similarity
        similarity = queries @ (features / np.maximum(norms, 1e-12)).T

        for i in range(len(queries)):
            np.maximum.at(scores[i], codes_block, similarity[i])
            winners = np.isfinite(similarity[i]) & (similarity[i] == scores[i, codes_block])
            image_rows[i, codes_block[winners]] = rows[winners]
    top = np.argsort(-scores)[:, :K]
    query_rows = np.arange(len(queries))[:, None]
    return image_rows[query_rows, top], scores[query_rows, top]

In [ ]:
def build_viewer_data():
    records, images, queries, metadata, embeddings, cutouts, names, image_dir = prepare_composites()
    neighbor_rows, similarities = find_neighbors(queries, metadata, embeddings, cutouts)
    assets, rows = {}, []

    def thumbnail(image):
        image.thumbnail((320, 240))
        return image

    for i, r in records.iterrows():
        key = f"composite-{i}"
        assets[key] = thumbnail(Image.fromarray(images[i].permute(1, 2, 0).numpy()))
        neighbors = []
        for rank, neighbor_row in enumerate(neighbor_rows[i]):
            neighbor = metadata.iloc[neighbor_row]
            filename = neighbor.filename
            if filename not in assets:  # only load if not loaded already
                with Image.open(image_dir / filename) as image:
                    assets[filename] = thumbnail(image.convert("RGB"))
            neighbors.append(dict(image=filename, label=names[neighbor.label],
                prediction=names[neighbor.prediction],
                similarity=similarities[i, rank]))
        rows.append(dict(species=names[r.y], variant=r.variant, background=r.bg_id,
            habitat=r.habitat, split=r.split, image=key, prediction=names[r.pred_y], neighbors=neighbors))
    return rows, assets

In [ ]:
rows, assets = build_viewer_data()

In [ ]:
BLUE, ORANGE, GREY = "#2563b0", "#b65e13", "#747b83"

def species_color(label, selected_species):
    if label.casefold() == "empty":
        return GREY
    return BLUE if label == selected_species else ORANGE

def draw(selected_rows):
    fig, axes = plt.subplots(len(selected_rows), 6, figsize=(14.4, 2.05 * len(selected_rows) + 0.55),
                             squeeze=False)
    fig.subplots_adjust(left=0.01, right=0.99, bottom=0.025, top=0.91, wspace=0.10, hspace=0.13)
    fig.text(0.01, 0.96, "Background comparison", fontsize=12, weight="bold", color="#27313a")
    fig.text(0.54, 0.96, "Frame = predicted species", color="#687681", fontsize=8.5)
    for x, label, color in [(0.71, "Selected species", BLUE), (0.82, "Other species", ORANGE),
                             (0.90, "Empty", GREY)]:
        fig.text(x, 0.96, label, color=color, fontsize=9, ha="center",
                 bbox=dict(boxstyle="square,pad=0.4", facecolor="white", edgecolor=color, linewidth=1.4))
    for row, r in enumerate(selected_rows):
        cards = [dict(image=r["image"], label=r["species"], prediction=r["prediction"])] + r["neighbors"]
        for col, card in enumerate(cards):
            ax = axes[row, col]
            ax.set(xlim=(0, 1), ylim=(0, 1))
            ax.axis("off")
            color = species_color(card["prediction"], r["species"])
            ax.add_patch(Rectangle((0, 0), 1, 1, facecolor="white",
                                   edgecolor=color, linewidth=1.6, clip_on=False))
            title = "COMPOSITE" if col == 0 else f"NEIGHBOR {col:02d}"
            detail = (f"Background {r['background']} · {r['habitat']}" if col == 0 else
                      f"Similarity {card['similarity']:.3f}")
            ax.text(0.045, 0.94, title, fontsize=8, weight="bold", color="#34414c", va="center")
            ax.text(0.045, 0.85, detail, fontsize=7.5, color=GREY, va="center")

            image = ImageOps.pad(assets[card["image"]], (360, 210), color="white")
            ax.imshow(image, extent=(0.025, 0.975, 0.245, 0.79), aspect="auto", zorder=2)
            for y, prefix, label in [(0.16, "True", card["label"]),
                                     (0.065, "Pred", card["prediction"])]:
                ax.text(0.045, y, f"{prefix}: {label}", fontsize=8,
                        color=species_color(label, r["species"]), va="center")
    plt.show()
    return fig

In [ ]:
species = widgets.Dropdown(options=sorted({r["species"] for r in rows}), description="Species:",
                           layout=widgets.Layout(width="320px"))
cutout = widgets.Dropdown(description="Cut-out:",
                         layout=widgets.Layout(width="155px"), style={"description_width": "55px"})
page = widgets.BoundedIntText(value=1, min=1, max=1)
previous = widgets.Button(description="<", tooltip="Previous backgrounds",
                          layout=widgets.Layout(width="34px", height="30px"))
following = widgets.Button(description=">", tooltip="Next backgrounds",
                           layout=widgets.Layout(width="34px", height="30px"))
page_label = widgets.Label(layout=widgets.Layout(width="78px"))
output = widgets.Output(layout=widgets.Layout(width="100%", overflow="visible"))
PER_PAGE = 4

def matching_rows():
    return sorted([r for r in rows if r["species"] == species.value
                   and r["variant"] == cutout.value],
                  key=lambda r: (r["split"], r["habitat"], r["background"]))

def render(_=None):
    selected = matching_rows()
    page.max = (len(selected) + PER_PAGE - 1) // PER_PAGE
    page_label.value = f"Page {page.value} / {page.max}"
    previous.disabled = page.value == 1
    following.disabled = page.value == page.max
    with output:
        clear_output(wait=True)
        start = (page.value - 1) * PER_PAGE
        fig = draw(selected[start:start + PER_PAGE])
        plt.close(fig)

def choose_species(_=None):
    cutout.options = sorted({r["variant"] for r in rows if r["species"] == species.value})
    cutout.value = cutout.options[0]
    page.value = 1
    render()

In [ ]:
species.observe(choose_species, names="value")

for control in [cutout, page]:
    control.observe(render, names="value")

previous.on_click(lambda button: setattr(page, "value", page.value - 1))
following.on_click(lambda button: setattr(page, "value", page.value + 1))

toolbar = widgets.HBox([species, cutout, previous, page_label, following],
                       layout=widgets.Layout(align_items="center", flex_flow="row wrap", grid_gap="8px"))
display(widgets.VBox([toolbar, output],
                     layout=widgets.Layout(width="100%", max_width="1440px", grid_gap="8px")))

choose_species()